# Embed citation contexts on Colab GPU

Runs the bulk embed pass on a Colab GPU against your local Postgres (exposed via `ngrok tcp 25432`), then rebuilds the HNSW index — fully self-contained.

This run embeds whatever is in `citation_contexts.context_text_for_embedding`. For the window-text experiment that column has already been repointed on the laptop (`scripts.prepare_window_reembed prepare` + `cutover`) to the **local window** (the citing sentence plus its neighbours), so no flag or code change is needed here.

**Before running:** `Runtime -> Change runtime type -> GPU`. The only cell you must edit is the connection cell below (ngrok host/port).

## 1. Fill in connection details

Paste the host/port from the `ngrok tcp 25432` window on your laptop, and your Postgres credentials (the ones in `.env`).

In [ ]:
NGROK_HOST = "4.tcp.eu.ngrok.io"   # <-- replace
NGROK_PORT = 15165                   # <-- replace
POSTGRES_USER = "mario"          # match your .env
POSTGRES_PASSWORD = "10diploma10"            # match your .env
POSTGRES_DB = "papers_db"               # match your .env

REPO_URL = "https://github.com/mariosam23/missing-citations-identifier"  # or use Files → Upload to copy the repo
BRANCH = "new_implementation"
BATCH_SIZE = 16  # T4 safe; can try 32 once kernel-side model is freed

import os
os.environ["DB_URL"] = (
    f"postgresql+psycopg://{POSTGRES_USER}:{POSTGRES_PASSWORD}"
    f"@{NGROK_HOST}:{NGROK_PORT}/{POSTGRES_DB}"
)
os.environ["EMBEDDER_MODEL_NAME"] = "BAAI/bge-large-en-v1.5"
os.environ["EMBEDDER_DIM"] = "1024"
os.environ["EMBEDDER_BATCH_SIZE"] = str(BATCH_SIZE)
os.environ["EMBEDDER_DEVICE"] = "cuda"
print("DB target:", os.environ["DB_URL"].split('@')[1])

## 2. Clone repo and install dependencies

Colab ships Python 3.11; the project pins 3.13 in `pyproject.toml`. We install the runtime deps directly (skip the strict `pip install -e .` path so the Python-version check does not block us). Nothing in `embed_contexts.py` / `embedder.py` uses 3.13-only syntax.

In [ ]:
!git clone --depth 1 --branch {BRANCH} {REPO_URL} /content/repo
%cd /content/repo
!pip install -q --upgrade pip
!pip install -q \
    'sentence-transformers>=3.2' 'torch>=2.4' \
    'sqlalchemy>=2.0' 'psycopg[binary]>=3.2' 'pgvector>=0.3.6' \
    'pydantic>=2.7' 'pydantic-settings>=2.4' \
    'typer>=0.12' 'tqdm>=4.66' xformers einops
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

## 3. Smoke test: load bge-large, encode a few sentences

Confirm the model loads at the right dim and that vectors come out unit-norm. VRAM after load is ~1.3 GB at fp16.

In [ ]:
import sys
sys.path.insert(0, '/content/repo/src')
import numpy as np
from pipeline.embedding.embedder import encode_texts, get_embedder

_ = get_embedder()
samples = [
    'We use BERT to encode the input tokens.',
    'Following Devlin et al., we apply masked language modelling.',
    'The transformer architecture relies on self-attention.',
    'Adam optimizer is used with a learning rate of 1e-4.',
    'We evaluate on the GLUE benchmark.',
    'Convolutional neural networks excel at image classification.',
    'Reinforcement learning from human feedback aligns the model with preferences.',
    'Layer normalization stabilises training.',
]
vectors = encode_texts(samples)
print('shape:', vectors.shape)
print('mean L2 norm:', float(np.linalg.norm(vectors, axis=1).mean()))

# Free the model from this kernel so the embed subprocess gets a clean GPU.
# Without this, both the kernel and the script hold a copy of bge-large and the GPU can OOM.
import gc, torch
import pipeline.embedding.embedder as _emb
_emb._model = None
gc.collect()
torch.cuda.empty_cache()
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv


## 4. Run the bulk embed pass

Resumable — if Colab disconnects you, just re-run this cell. The anti-join inside `embed_contexts.py` skips rows that already have an embedding.

In [ ]:
!cd /content/repo && PYTORCH_ALLOC_CONF=expandable_segments:True PYTHONPATH=src python -m scripts.embed_contexts --batch-size {BATCH_SIZE}


## 5. Rebuild the HNSW index, then sanity-check

The next cell rebuilds the HNSW cosine index over the freshly-inserted vectors by calling `scripts.prepare_window_reembed finalize` (same params as before: `m=16, ef_construction=64`). The cell after verifies the row count, an example vector's L2 norm, and that the index exists.

No laptop step is needed. When both finish, tell the assistant — it will A/B the window vs. sentence embeddings on val and keep or roll back.

In [ ]:
!cd /content/repo && PYTHONPATH=src python -m scripts.prepare_window_reembed finalize

In [ ]:
from sqlalchemy import text
from database.postgres.engine import get_session
import numpy as np

with get_session() as session:
    n = session.execute(text('SELECT COUNT(*) FROM citation_context_embeddings')).scalar_one()
    has_idx = session.execute(
        text("SELECT to_regclass('public.citation_context_embedding_hnsw_idx') IS NOT NULL")
    ).scalar_one()
    row = session.execute(
        text('SELECT context_id, embedding FROM citation_context_embeddings LIMIT 1')
    ).first()
    print('rows:', n, '| hnsw index present:', has_idx)
    if row is not None:
        vec = np.asarray(row.embedding, dtype=np.float32)
        print('first row dim:', vec.shape[0], 'L2:', float(np.linalg.norm(vec)))
